In [39]:
# ========== 安装依赖：Gradio + 表格 + Hugging Face 本地推理栈 ==========
# -q 安静安装；transformers/torch/accelerate/bitsandbytes 用于本地 4bit 因果 LM；
# python-dotenv 用于从 .env 读 HUGGINGFACE_TOKEN
!pip install -q gradio pandas transformers torch accelerate bitsandbytes python-dotenv


zsh:1: command not found: pip


In [ ]:
# ========== 导入 + 尼日利亚合成数据用的常量池 ==========
# 标准库：环境变量、JSON、随机、时间戳工具
import os
import json
import random
from datetime import datetime

# PyTorch：本地模型张量与 device
import torch
# pandas / numpy：表格与数值；Gradio：Web UI
import pandas as pd
import numpy as np
import gradio as gr
# Hugging Face：因果语言模型、分词器、4bit 量化配置
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# huggingface_hub.login：也可用 token 登录（本格主要靠 dotenv）
from huggingface_hub import login
# load_dotenv：把 .env 读进环境变量
from dotenv import load_dotenv

# 加载 .env（例如 HUGGINGFACE_TOKEN），不要把密钥写进笔记本
load_dotenv()

# —— 常数：随机生成客户/商家时抽用的英文词表（保持原字符串，影响生成内容）——
# 银行名候选
BANKS = ["Access", "GTBank", "Opay", "First", "Zenith", "Fidelity"]
# 州/城候选
STATES = ["Lagos", "Abuja", "Kano", "Rivers", "Oyo", "Kaduna", "Delta", "Enugu"]
# 名 / 姓候选
FIRST_NAMES = ["Ade", "Chidi", "Musa", "Olu", "Ngozi", "Femi", "Zainab", "Emeka"]
LAST_NAMES = ["Okafor", "Balogun", "Abubakar", "Okonkwo", "Adebayo", "Musa"]
# 商家类型候选
BUSINESSES = ["Restaurant", "Fashion", "Tech", "Agriculture", "Retail"]


In [41]:
# ========== NigerianGenerator：纯随机规则生成客户/商家字典 ==========
# 小工具类：不调 LLM，靠 random.choice 拼出结构化假数据（可作 LLM 失败时的回退）
class NigerianGenerator:
    # 手机号：070/080/090 前缀 + 8 位随机数字
    def phone(self): return random.choice(["070","080","090"]) + ''.join([str(random.randint(0,9)) for _ in range(8)])
    # 姓名：名 + 姓
    def name(self): return f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
    # 客户记录：name / phone / state / bank
    def customer(self): return {"name": self.name(), "phone": self.phone(), "state": random.choice(STATES), "bank": random.choice(BANKS)}
    # 商家记录：品牌名、类型、电话、州
    def business(self): return {"name": f"{random.choice(['Prime','Royal'])} {random.choice(BUSINESSES)}", "type": random.choice(BUSINESSES), "phone": self.phone(), "state": random.choice(STATES)}

# 实例化一份全局生成器，后面 generate / create_dataset 会用到
gen = NigerianGenerator()


In [46]:
# ========== 从 .env 取 HF token，加载选定的本地 Instruct 模型（4bit）==========
# 再次导入：本格可单独重跑时自洽（逻辑与标识符保持原样）
import os
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 重新加载 .env，确保 HUGGINGFACE_TOKEN 可用
load_dotenv()
# 读 Hugging Face token；gated 模型（如 Llama）需要登录权限
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

# 可用型号：界面显示名 → Hub 上的 model id（字符串勿改）
models = {
    "Phi-4-mini (Fast)": "microsoft/Phi-4-mini-instruct",
    "Llama-3.2-3B (Balanced)": "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen-Coder (Code)": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Phi-3.5 (Legacy)": "microsoft/Phi-3.5-mini-instruct"
}

# 选择型号（更改此行以使用不同的型号）；默认 Phi-4-mini
SELECTED_MODEL = models["Phi-4-mini (Fast)"]  # Default

try:
    # 加载分词器；token= 用于访问需授权的仓库
    tokenizer = AutoTokenizer.from_pretrained(SELECTED_MODEL, token=HF_TOKEN)
    # 加载因果 LM：4bit 量化 + device_map=auto 自动分配 GPU/CPU
    model = AutoModelForCausalLM.from_pretrained(
        SELECTED_MODEL,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True),
        device_map="auto",
        token=HF_TOKEN
    )
    # 成功提示（文案保持原样）
    print(f"✅ Loaded: {SELECTED_MODEL}")
except Exception as e:
    # 失败则打印原因，并把 model 置 None，后面 generate 会回退到规则生成
    print(f"⚠️ Failed to load {SELECTED_MODEL}: {e}")
    model = None


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

⚠️ Failed to load microsoft/Phi-4-mini-instruct: No package metadata was found for bitsandbytes


In [47]:
# ========== LLM 生成 + 按数据类型抽一条英文 prompt ==========
# 用本地模型按 prompt 生成文本；若 model 未加载则退回规则客户字典
def generate(prompt):
    # 模型不可用：直接返回随机客户，保证流水线不崩
    if not model: return gen.customer()

    # 组装 chat messages（单条 user）
    messages = [{"role": "user", "content": prompt}]
    # 用模型自带 chat template 转成纯文本（tokenize=False 先不转 id）
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    # 编码成张量并搬到模型所在 device
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 贪心/默认采样生成，最多 150 个新 token
    outputs = model.generate(**inputs, max_new_tokens=150)
    # 把整段输出（含 prompt）解码回字符串
    return tokenizer.decode(outputs[0])

# 测试每个模型的不同提示：按 Customers / Businesses 随机抽一条英文 prompt 再 generate
def generate_diverse(dtype):
    # 提示词字典：键是数据类型，值是若干英文 prompt（内容保持原样）
    prompts = {
        "Customers": [
            "Generate a Lagos businessman",
            "Generate a young Nigerian professional",
            "Generate a Nigerian market trader"
        ],
        "Businesses": [
            "Generate a Nigerian tech startup",
            "Generate a Lagos restaurant",
            "Generate a Nigerian fashion brand"
        ]
    }
    # 随机选一条 prompt，交给 generate
    return generate(random.choice(prompts[dtype]))


In [48]:
# ========== 组装数据集：先规则填满，可选每隔一条用 LLM 覆盖 ==========
# dtype：Customers/Businesses；n：条数；use_llm：是否混入本地模型输出
def create_dataset(dtype, n, use_llm):
    # 先用规则生成器填满 n 条（客户或商家）
    data = [gen.customer() if dtype=="Customers" else gen.business() for _ in range(n)]

    # 若勾选 Use LLM 且模型已加载：每隔一条（步长 2）用 LLM 结果覆盖
    if use_llm and model:
        for i in range(0, n, 2):  # Every other record
            data[i] = generate_diverse(dtype)

    # list → DataFrame
    df = pd.DataFrame(data)
    # 在最左插入 id 列：NG-0001 风格
    df.insert(0, "id", [f"NG-{i+1:04d}" for i in range(n)])
    # 返回完整表
    return df


In [49]:
# ========== Gradio Blocks：类型 / 条数 / 是否用 LLM → 表格 ==========
# 搭一个简单界面：左侧控件，右侧 Dataframe；share=True 尝试生成公网链接
with gr.Blocks(title="Nigerian Data Generator") as demo:
    # 页面标题（Markdown，含国旗 emoji，文案保持原样）
    gr.Markdown("# 🇳🇬 Nigerian Data Generator")

    # 一行两列：左控件、右输出
    with gr.Row():
        with gr.Column():
            # 数据类型下拉
            dtype = gr.Dropdown(["Customers", "Businesses"], label="Type")
            # 记录数滑块：1~50，默认 10
            n = gr.Slider(1, 50, 10, label="Records")
            # 是否启用本地 LLM 混入
            llm = gr.Checkbox(label="Use LLM", value=True)
            # 主按钮
            btn = gr.Button("Generate", variant="primary")

        # 输出表格（占右侧）
        out = gr.Dataframe()

    # 点击：lambda 把滑块值转 int，再调用 create_dataset
    btn.click(lambda d,n,l: create_dataset(d,int(n),l), [dtype,n,llm], out)

# 启动；share=True 会尝试 Gradio 临时公网隧道
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://018457bb6ab9ca15bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
